# Part H: Audio Feature Extraction

---

## The Goal: From Raw Audio to Something Useful

You already used `torchaudio.transforms.MelSpectrogram` in Part B and got 74% accuracy. But you used it as a black box — call the transform, get a tensor, done.

Part H is about understanding what's actually happening inside that black box, step by step. This matters because when your EEND system misbehaves, you need to know whether the problem is in the feature extraction or the model. You can't debug a black box.

There are four steps that turn a raw waveform into Log-Mel-Filterbank features:

```
Raw waveform  →  [1] Framing  →  [2] STFT  →  [3] Mel filterbank  →  [4] Log
(48000,)          (N, 400)        (N, 201)       (N, 80)               (N, 80)
```

where N is number of frames \
Let's go through each one.

---

## Step 1: Framing

You can't apply a Fourier Transform to 48,000 samples at once and get anything useful. The Fourier Transform assumes the signal is **stationary** — that its frequency content doesn't change over the analysis window. Speech is emphatically not stationary over 3 seconds: a person might say "ba-na-na" and the frequencies change dramatically between each phoneme.

The solution: cut the signal into short overlapping **frames**, each short enough that the frequency content is approximately constant within it.

```
Frame length: 400 samples = 25ms  ← one "snapshot" of frequency content
Hop length:   160 samples = 10ms  ← how far to slide before next snapshot
```

Why overlap? Because if you slide by exactly 400 samples each time (no overlap), you'd miss transitions that happen at frame boundaries. Overlapping ensures every moment in time is well-represented in at least one frame.

For a 48,000 sample signal:
```
Number of frames = floor((48000 - 400) / 160) + 1 = 298 frames
```

Each frame is a 400-sample window — a snapshot of the audio at that moment in time.

---

## Step 2: STFT — What Frequencies Are in This Frame?

Given one 400-sample frame, the **Discrete Fourier Transform** decomposes it into a sum of sinusoids and tells you the amplitude of each frequency component.

You don't need to understand the math of the DFT. What matters is:

- Input: 400 samples (one frame)
- Output: 201 complex numbers — each one represents one frequency bin
- The 201 frequency bins span from 0 Hz to 8000 Hz (Nyquist = sample_rate / 2 = 16000 / 2)
- Each complex number has a magnitude (how much energy at this frequency) and phase (timing)

For speech, **magnitude is what matters**. Phase carries information about the exact timing of sound waves, which isn't useful for recognition or diarization. So you take `|complex number|` and discard the phase.

Why 201 bins? For an n_fft of 400, the output has `n_fft // 2 + 1 = 201` unique frequency bins. This comes from the symmetry of the Fourier Transform on real-valued signals — the second half of the output is always a mirror of the first half, so you only keep the first half.

After applying STFT to all frames:
```
Shape: (201 freq bins, 298 time frames)
Each cell: magnitude of energy at frequency f, at time frame t
```

This is your **spectrogram**.

---

## Step 3: Mel Filterbank — Perceptual Frequency Warping

You now have 201 evenly-spaced frequency bins from 0 to 8000 Hz. The problem: humans don't perceive frequency linearly.

The difference between 100 Hz and 200 Hz sounds like a large interval to your ears — it's roughly one octave for low voices. The difference between 7900 Hz and 8000 Hz sounds tiny — both are very high-pitched, barely distinguishable.

The **mel scale** converts Hz to a perceptual scale where equal intervals sound equally far apart. The conversion is approximately:

```
mel = 2595 × log10(1 + Hz / 700)
```

You don't need to memorize this. The key shape: the mel scale is logarithmic at high frequencies and roughly linear at low frequencies. It heavily compresses the high-frequency bins.

A **mel filterbank** is a set of triangular filters — typically 80 of them — spread evenly on the mel scale. Each filter covers a range of frequency bins and outputs a weighted sum of the energy in that range:

```
Linear (Hz):  |--|--|--|--|--|--|--|--|--|--|  201 bins, equal spacing
Mel scale:    |-|-|-|--|--|---|----|----|---|  80 bins, dense at low freq
```

Applying all 80 filters to the 201-bin spectrogram gives you:
```
Input:  (201, 298)  ← raw spectrogram
Output: (80, 298)   ← mel spectrogram
```

You've gone from 201 bins to 80, and the new bins are perceptually meaningful. Each mel bin roughly corresponds to how a human ear would perceive that frequency region.

---

## Step 4: Log — Compressing the Dynamic Range

The mel filterbank outputs raw energy values. Speech energy spans several orders of magnitude:

```
Silence:       0.00001
Soft speech:   0.01
Loud speech:   10.0
```

If you feed these raw values into a neural network, the loud speech samples completely dominate every gradient update. The network effectively ignores quiet speech.

Taking the log compresses this range:
```
log(0.00001) = -11.5
log(0.01)    = -4.6
log(10.0)    =  2.3
```

Now the range is roughly -12 to +3 — manageable for gradient-based training. You add a small epsilon before the log to avoid `log(0) = -∞`:

```python
log_mel = torch.log(mel + 1e-8)
```

The final output — Log-Mel-Filterbank features — has shape `(80, 298)` for a 3-second clip. This is exactly what EEND takes as input.

---

## Why 80 Mel Bins?

The honest answer: it's a convention that has worked well empirically across decades of speech research, established before deep learning.

The reasoning behind it:

- Human speech has most of its phonetically relevant content below 8000 Hz
- The mel scale has roughly 80 "just-noticeable difference" steps in that range — frequency differences that a human can reliably distinguish
- 80 bins captures essentially all phonetically relevant frequency variation
- 40 bins loses some detail (works but slightly worse)
- 128 bins adds redundancy without meaningful gain for speech

For speaker diarization specifically, voice characteristics (pitch, timbre) are well-captured at 80 bins. This is why it became the standard for EEND and most speech recognition systems.

---

## Frame Length and Hop Length: How to Choose

These two parameters control the time-frequency tradeoff:

**Frame length (n_fft):**
```
Longer frame → better frequency resolution, worse time resolution
Shorter frame → better time resolution, worse frequency resolution
```

This is the **uncertainty principle** for signals — you can't have perfect resolution in both time and frequency simultaneously. A 25ms frame is a compromise:
- Long enough to capture ~4 pitch periods (enough for frequency analysis)
- Short enough that the frequency content is approximately stable within the frame

**Hop length:**
```
Smaller hop → more frames, more overlap, smoother representation, slower
Larger hop  → fewer frames, less overlap, coarser time resolution, faster
```

10ms hop (160 samples) is standard because:
- It gives ~100 frames per second — enough time resolution to capture phoneme boundaries (~50-100ms)
- It matches how speech recognition systems are evaluated (10ms frame rate)

**For your EEND system specifically:**
- Your clips are 3 seconds = 48,000 samples
- With frame=400, hop=160: you get 298 time frames
- That's 298 "snapshots" of frequency content across 3 seconds
- Each snapshot is a vector of 80 mel values
- Final shape: `(80, 298)` per clip

---

## The Full Pipeline as Code (Conceptually)

```t
Raw audio: (48000,)
    ↓  frame into overlapping windows
Frames: (298, 400)
    ↓  apply DFT to each frame
STFT magnitude: (298, 201)
    ↓  apply 80 mel triangular filters
Mel spectrogram: (298, 80)
    ↓  take log
Log-Mel-Filterbank: (298, 80)
    ↓  transpose to standard convention
LMF features: (80, 298)   ← this is what you feed to EEND
```

`torchaudio.transforms.MelSpectrogram` does steps 1-3 in one call. You add step 4 manually with `torch.log`.

---

## TODO: Build the LMF Pipeline from Scratch

**Goal:** Implement Log-Mel-Filterbank extraction *manually* using lower-level functions, so you understand what `MelSpectrogram` is doing inside. Then verify your manual version matches `torchaudio`'s output exactly.

**Requirements:**

**Part 1 — Manual extraction:**

Implement LMF extraction using only these building blocks (in order):
- `torch.stft()` — applies DFT to overlapping frames
- `torch.abs()` — extract magnitude from complex output
- `torchaudio.functional.melscale_fbanks()` — get the 80 mel triangular filters
- Matrix multiplication to apply filters to the spectrogram
- `torch.log()` with epsilon

Each step should print the intermediate tensor shape so you can see the pipeline.

**Part 2 — Verification:**

Run the same audio through `torchaudio.transforms.MelSpectrogram` (as you did in Part B), apply log, and compare with your manual output using `torch.allclose(atol=1e-5)`.

**Part 3 — Experiment:**

Plot side by side:
- Frame length 200 (12.5ms) vs 400 (25ms) vs 800 (50ms)
- Keep hop_length=160 fixed

For each, print the output shape and plot the LMF as a heatmap. What do you notice about the frequency resolution vs time resolution tradeoff visually?

**Hints:**
- `torch.stft()` requires `return_complex=True` — the output shape is `(freq_bins, time_frames)` as complex numbers
- `torch.abs()` on a complex tensor gives you the magnitude
- `torchaudio.functional.melscale_fbanks(n_freqs=n_fft//2+1, f_min=0, f_max=8000, n_mels=80, sample_rate=16000)` returns a `(n_freqs, n_mels)` matrix
- To apply the filterbank: `mel_spec = torch.matmul(filterbank.T, magnitude)` — think carefully about which dimension you're summing over
- `torch.stft()` needs a `window` argument — use `torch.hann_window(n_fft)`
- Your audio going into `torch.stft()` should be shape `(time,)` — squeeze out the channel dim first

**Key questions:**
- At what step does the shape change most dramatically (largest reduction in size)?
- With frame_length=200, your frequency bins only go up to a certain resolution. How many Hz does each bin represent? Does this affect how well you can distinguish speech sounds?
- Why do we use a Hann window instead of just taking raw frames?

---

Work through Part 1 first — print the shape at every single step and show me the output before moving to verification.

In [260]:
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torchaudio.transforms as T
import torchaudio
import math
from pathlib import Path
import os
import sys
import json
import matplotlib.pyplot as plt

# Check if __file__ exists (it won't in Jupyter)
try:
    current_dir = Path(__file__).parent
except NameError:
    # If in Jupyter, use the current working directory
    current_dir = Path(os.getcwd())

# Add project root to Python path
project_root = current_dir.parent.parent.parent
sys.path.insert(0, str(project_root))

from src.preprocessing.audio_utils import load_audio


# Device configuration (for your MacBook)
if torch.backends.mps.is_available():
    device = torch.device('mps')
    print("✅ Using Apple Silicon GPU")
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print("✅ Using NVIDIA GPU")
else:
    device = torch.device('cpu')
    print("⚠️ Using CPU")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")


✅ Using Apple Silicon GPU
PyTorch version: 2.10.0
Device: mps


In [261]:
class SpeakerDataset(torch.utils.data.Dataset):
    def __init__(self, manifest_path, limit=None):
        # Load manifest JSON
        # Store all entries as self.data
        with open(manifest_path, "r") as f:
            full_data = json.load(f)
        
        if limit:
            self.data = full_data[:limit]
        else:
            self.data = full_data
    
    def __len__(self):
        # Return number of samples
        return len(self.data)
    
    def __getitem__(self, idx):
        # Get entry at index idx
        # Load audio from disk
        # Extract features
        # Get label (num_speakers - 1)
        # Return (features, label)
        entry = self.data[idx]
        mixture_path = Path(entry['mixture_path'])
        mixture_audio, _ = load_audio(mixture_path)
        mixture_tensor = torch.from_numpy(mixture_audio)
        
        max_len = 480000 # 30 seconds at 16kHz
        
        if mixture_tensor.size(0) > max_len:
            # Truncate if too long
            mixture_tensor = mixture_tensor[:max_len]
        else:
            # Pad with zeros if too short
            padding = max_len - mixture_tensor.size(0)
            mixture_tensor = torch.nn.functional.pad(mixture_tensor, (0, padding))
        
        mixture_tensor = mixture_tensor.unsqueeze(0)
        
        speaker_count = int(entry['num_speakers']) - 1
        label_tensor = torch.tensor(speaker_count, dtype=torch.long)
        
        return mixture_tensor, label_tensor

In [262]:
train_manifest_path = Path.cwd().parent.parent.parent / "data" / "processed" / "train" / "train_manifest.json"
train_dataset = SpeakerDataset(train_manifest_path)
print(len(train_dataset))

val_manifest_path = Path.cwd().parent.parent.parent / "data" / "processed" / "val" / "val_manifest.json"
val_dataset = SpeakerDataset(val_manifest_path, 2000)
print(len(val_dataset))

10000
2000


In [263]:
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    drop_last=True
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    drop_last=False
)

In [264]:
audio = train_dataset[0][0][0]
print(len(audio))

480000


### todo 1

The error is telling you everything — your manual pipeline produces 3001 time frames and torchaudio produces 2998. Same audio, same n_fft, same hop_length, but different frame counts. That means one of them is padding the audio before framing and the other isn't.

The culprit is `torch.stft()`. By default it pads the signal on both sides so the first frame is centered on sample 0. This adds `n_fft // 2 = 200` samples on each side before framing, which produces more frames than you expect.

You have two options to fix this — think about which one makes more sense before applying it:

**Option A:** Tell `torch.stft()` not to center the frames by passing `center=False`

**Option B:** Tell torchaudio to center its frames to match what `torch.stft()` does by default.

**Option C:** Tell `torchaudio` and `torch.stft()` either to have `center=False` or `center=True`


One more thing to check once the frame counts match: your audio is 27-30 seconds long, but your model expects 48,000 samples (3 seconds). Are you slicing the audio before passing it in, or passing the full long clip? The shape mismatch will be obvious once you print `audio.shape` before the function call.

In [265]:
def manualLMF(audio, n_fft, hop_length):
    dft = torch.stft(
        audio, 
        n_fft, 
        hop_length, 
        return_complex=True,
        window=torch.hann_window(n_fft),
        center=False
    )    
    magnitude = torch.abs(dft)
    filterbank = torchaudio.functional.melscale_fbanks(
        n_freqs=n_fft // 2 + 1, 
        f_min=0, 
        f_max=8000, 
        n_mels=80, 
        sample_rate=16000
    )
    mel_spec = torch.matmul(filterbank.T, magnitude)
    log = torch.log(mel_spec + 1e-8)
    return log

def torchLMF(audio, n_fft, hop_length):
    torch_lmf = T.MelSpectrogram(
        sample_rate=16000,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=80,
        center=False,
        power=1.0
        )
    mel_spec = torch_lmf(audio)
    log = torch.log(mel_spec + 1e-8)
    return log



In [266]:
n_fft = 400
hop_length = 160

manualLMF_extracted = manualLMF(audio, n_fft, hop_length)
torchLMF_extractd = torchLMF(audio, n_fft, hop_length)

print(torch.allclose(manualLMF_extracted, torchLMF_extractd, atol=1e-5))

True
